Organizar em modelo estrela com uma tabela fato(`fato_reclamacao`) e quatro dimensões: empresa, local, problema e consumidor.

In [0]:
%sql
-- Lista cada empresa uma única vez e dá um número (id) para cada uma
CREATE OR REPLACE TABLE workspace.consumidor.dim_empresa AS
SELECT
  ROW_NUMBER() OVER (ORDER BY nome_fantasia) AS id_empresa,
  nome_fantasia,
  segmento_mercado
FROM (
  SELECT DISTINCT nome_fantasia, segmento_mercado
  FROM workspace.consumidor.silver_reclamacoes
);

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.consumidor.dim_local AS
SELECT
  ROW_NUMBER() OVER (ORDER BY regiao, uf, cidade) AS id_local,
  regiao, uf, cidade
FROM (
  SELECT DISTINCT regiao, uf, cidade
  FROM workspace.consumidor.silver_reclamacoes
);

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.consumidor.dim_problema AS
SELECT
  ROW_NUMBER() OVER (ORDER BY area, assunto, grupo_problema, problema) AS id_problema,
  area, assunto, grupo_problema, problema
FROM (
  SELECT DISTINCT area, assunto, grupo_problema, problema
  FROM workspace.consumidor.silver_reclamacoes
);

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.consumidor.dim_consumidor AS
SELECT
  ROW_NUMBER() OVER (ORDER BY sexo, faixa_etaria) AS id_consumidor,
  sexo, faixa_etaria
FROM (
  SELECT DISTINCT sexo, faixa_etaria
  FROM workspace.consumidor.silver_reclamacoes
);

In [0]:
%sql
-- Define a pasta de trabalho (catálogo workspace, schema consumidor)
USE workspace.consumidor;

CREATE OR REPLACE TABLE fato_reclamacao AS
SELECT
  dim_empresa.id_empresa,
  dim_local.id_local,
  dim_problema.id_problema,
  dim_consumidor.id_consumidor,
  silver_reclamacoes.data_finalizacao,
  silver_reclamacoes.tempo_resposta,
  silver_reclamacoes.respondida,
  silver_reclamacoes.procurou_empresa,
  silver_reclamacoes.como_comprou_contratou,
  silver_reclamacoes.situacao,
  silver_reclamacoes.avaliacao_reclamacao,
  silver_reclamacoes.nota_consumidor
FROM silver_reclamacoes
JOIN dim_empresa
  ON silver_reclamacoes.nome_fantasia = dim_empresa.nome_fantasia
  AND silver_reclamacoes.segmento_mercado = dim_empresa.segmento_mercado
JOIN dim_local
  ON silver_reclamacoes.regiao = dim_local.regiao
  AND silver_reclamacoes.uf = dim_local.uf
  AND silver_reclamacoes.cidade = dim_local.cidade
JOIN dim_problema
  ON silver_reclamacoes.area = dim_problema.area
  AND silver_reclamacoes.assunto = dim_problema.assunto
  AND silver_reclamacoes.grupo_problema = dim_problema.grupo_problema
  AND silver_reclamacoes.problema = dim_problema.problema
JOIN dim_consumidor
  ON silver_reclamacoes.sexo = dim_consumidor.sexo
  AND silver_reclamacoes.faixa_etaria = dim_consumidor.faixa_etaria;

In [0]:
%sql
SELECT 'silver' AS tabela, COUNT(*) AS linhas FROM workspace.consumidor.silver_reclamacoes
UNION ALL SELECT 'fato_reclamacao', COUNT(*) FROM workspace.consumidor.fato_reclamacao
UNION ALL SELECT 'dim_empresa', COUNT(*) FROM workspace.consumidor.dim_empresa
UNION ALL SELECT 'dim_local', COUNT(*) FROM workspace.consumidor.dim_local
UNION ALL SELECT 'dim_problema', COUNT(*) FROM workspace.consumidor.dim_problema
UNION ALL SELECT 'dim_consumidor', COUNT(*) FROM workspace.consumidor.dim_consumidor;

In [0]:
%sql
USE workspace.consumidor;

-- Chaves primárias: o id precisa ser obrigatório (NOT NULL) antes de virar PK
ALTER TABLE dim_empresa ALTER COLUMN id_empresa SET NOT NULL;
ALTER TABLE dim_empresa ADD CONSTRAINT pk_dim_empresa PRIMARY KEY (id_empresa);

ALTER TABLE dim_local ALTER COLUMN id_local SET NOT NULL;
ALTER TABLE dim_local ADD CONSTRAINT pk_dim_local PRIMARY KEY (id_local);

ALTER TABLE dim_problema ALTER COLUMN id_problema SET NOT NULL;
ALTER TABLE dim_problema ADD CONSTRAINT pk_dim_problema PRIMARY KEY (id_problema);

ALTER TABLE dim_consumidor ALTER COLUMN id_consumidor SET NOT NULL;
ALTER TABLE dim_consumidor ADD CONSTRAINT pk_dim_consumidor PRIMARY KEY (id_consumidor);

-- Chaves estrangeiras: ligam a fato a cada dimensão
ALTER TABLE fato_reclamacao ADD CONSTRAINT fk_fato_empresa
  FOREIGN KEY (id_empresa) REFERENCES dim_empresa (id_empresa);
ALTER TABLE fato_reclamacao ADD CONSTRAINT fk_fato_local
  FOREIGN KEY (id_local) REFERENCES dim_local (id_local);
ALTER TABLE fato_reclamacao ADD CONSTRAINT fk_fato_problema
  FOREIGN KEY (id_problema) REFERENCES dim_problema (id_problema);
ALTER TABLE fato_reclamacao ADD CONSTRAINT fk_fato_consumidor
  FOREIGN KEY (id_consumidor) REFERENCES dim_consumidor (id_consumidor);

Captura do modelo estrela na Catalog → workspace → consumidor → fato_reclamacao -> view relationships